# Analyse COS — Table brute + Table agregee
> Strategie memoire : on charge la table brute **par chunks** de 500K lignes  
> On filtre COS dans chaque chunk et on accumule uniquement les lignes utiles  
> Resultat final = meme taille que la table agregee, zéro crash kernel  
> Dataiku DSS · python36distrib_new

In [ ]:
# ── 0. Imports ──────────────────────────────────────────────────────────────
import dataiku
import pandas as pd
import numpy as np
import gc
import warnings
warnings.filterwarnings('ignore')

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import ipywidgets as widgets

print('Imports OK')

In [ ]:
# ── 1. Parametres ───────────────────────────────────────────────────────────

# --- Table brute (transactionnelle) ---
DATASET_BRUT      = 'retraits_transactionnel'   # <- MODIFIER
COL_AUTOMATE_B    = 'automates'
COL_MONTANT_B     = 'montant_ope'        # en centimes
COL_DATE_B        = 'date_operation'
COL_RESEAU_B      = 'reseau'
COL_TYPE_CRO_B    = 'type'
COL_PRIVATIVE_B   = 'type_carte_privative'
CODES_CRO         = ['AXA', '60']        # vrais retraits uniquement
CODE_RESEAU_COS   = '11'
CODE_PRIVAT_COS   = 'COS'
CHUNK_SIZE        = 500_000              # nb lignes par chunk

# --- Table agregee ---
DATASET_AGG       = 'fiche_identite_gab_mensuelle'   # <- MODIFIER si besoin
COL_AUTOMATE_A    = 'num_automate'
COL_NB_COS        = 'ret_nb_cos'
COL_CAP_COS       = 'cap_cos_nb'
COL_TAUX_COS      = 'taux_capture_cos_pct'
COL_MONTANT_MOY   = 'ret_montant_moyen'  # en centimes

# --- Annee analysee ---
ANNEE = 2025

print('Parametres OK')

## Partie 1 — Table brute (lecture par chunks)

In [ ]:
# ── 2. Lecture table brute par chunks ───────────────────────────────────────
# Principe : on lit 500K lignes a la fois, on filtre COS, on garde le resultat
# Memoire max utilisee = 1 chunk (~500K lignes) + les lignes COS accumulees

print('Lecture de la table brute par chunks de {:,} lignes...'.format(CHUNK_SIZE))

COLONNES_BRUT = [
    COL_AUTOMATE_B, COL_MONTANT_B, COL_DATE_B,
    COL_RESEAU_B, COL_TYPE_CRO_B, COL_PRIVATIVE_B,
]

ds_brut  = dataiku.Dataset(DATASET_BRUT)
reader   = ds_brut.get_dataframe_iterator(
    chunksize=CHUNK_SIZE,
    columns=COLONNES_BRUT,
)

chunks_cos = []
total_lus  = 0
total_cos  = 0

for i, chunk in enumerate(reader):
    total_lus += len(chunk)

    # Typage date
    chunk[COL_DATE_B] = pd.to_datetime(chunk[COL_DATE_B], errors='coerce')

    # Filtres en cascade (du plus restrictif au moins couteux)
    mask = (
        (chunk[COL_DATE_B].dt.year == ANNEE) &
        (chunk[COL_TYPE_CRO_B].isin(CODES_CRO)) &
        (
            (chunk[COL_RESEAU_B].astype(str).str.strip() == CODE_RESEAU_COS) |
            (chunk[COL_PRIVATIVE_B].astype(str).str.strip().str.upper() == CODE_PRIVAT_COS)
        )
    )
    cos_chunk = chunk[mask].copy()
    total_cos += len(cos_chunk)

    if len(cos_chunk) > 0:
        chunks_cos.append(cos_chunk)

    del chunk, cos_chunk
    gc.collect()

    if (i + 1) % 5 == 0:
        print('  Chunk {} — {:,} lignes lues au total | {:,} COS retenus'.format(
            i+1, total_lus, total_cos))

print()
print('Lecture terminee : {:,} lignes lues au total'.format(total_lus))
print('Lignes COS retenues : {:,}'.format(total_cos))

In [ ]:
# ── 3. Consolidation + enrichissement ───────────────────────────────────────
if len(chunks_cos) == 0:
    print('Aucune donnee COS trouvee — verifiez les parametres.')
else:
    df_brut_cos = pd.concat(chunks_cos, ignore_index=True)
    del chunks_cos
    gc.collect()

    # Renommage uniforme
    df_brut_cos = df_brut_cos.rename(columns={
        COL_AUTOMATE_B: 'num_automate',
        COL_MONTANT_B:  'montant_cts',
    })

    df_brut_cos['montant_eur'] = df_brut_cos['montant_cts'] / 100.0
    df_brut_cos['mois']        = df_brut_cos[COL_DATE_B].dt.month
    df_brut_cos['heure']       = df_brut_cos[COL_DATE_B].dt.hour

    NOMS_MOIS = {1:'Jan',2:'Fev',3:'Mar',4:'Avr',5:'Mai',6:'Jun',
                 7:'Jul',8:'Aou',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
    df_brut_cos['mois_nom'] = df_brut_cos['mois'].map(NOMS_MOIS)

    print('Table brute COS consolidee : {:,} lignes | {:,} GAB'.format(
        len(df_brut_cos), df_brut_cos['num_automate'].nunique()))
    display(df_brut_cos.head(3))

In [ ]:
# ── 4. Agregation depuis la table brute ─────────────────────────────────────
# On calcule les memes KPIs que la table agregee pour pouvoir comparer

agg_brut_mois = df_brut_cos.groupby(['mois','mois_nom']).agg(
    nb_cos   = ('montant_eur', 'count'),
    montant  = ('montant_eur', 'sum'),
    moy      = ('montant_eur', 'mean'),
    nb_gab   = ('num_automate','nunique'),
).reset_index().sort_values('mois')

agg_brut_gab = df_brut_cos.groupby('num_automate').agg(
    nb_cos      = ('montant_eur', 'count'),
    montant_tot = ('montant_eur', 'sum'),
    montant_moy = ('montant_eur', 'mean'),
    nb_mois     = ('mois',        'nunique'),
    nb_jours    = (COL_DATE_B,    'nunique'),
).reset_index().sort_values('nb_cos', ascending=False)

agg_brut_heure = df_brut_cos.groupby('heure').agg(
    nb_cos  = ('montant_eur', 'count'),
    montant = ('montant_eur', 'sum'),
).reset_index().sort_values('heure')

print('Agregations table brute OK')
print('  Par mois : {} lignes | Par GAB : {:,} | Par heure : {} lignes'.format(
    len(agg_brut_mois), len(agg_brut_gab), len(agg_brut_heure)))

## Partie 2 — Table agregee

In [ ]:
# ── 5. Lecture table agregee ────────────────────────────────────────────────
COLONNES_AGG = [
    COL_AUTOMATE_A, 'annee', 'mois',
    COL_NB_COS, COL_CAP_COS, COL_TAUX_COS, COL_MONTANT_MOY,
]

ds_agg   = dataiku.Dataset(DATASET_AGG)
df_agg   = ds_agg.get_dataframe(columns=COLONNES_AGG)

df_agg = df_agg[
    (df_agg['annee'] == ANNEE) &
    (df_agg[COL_NB_COS] > 0)
].copy()

df_agg['montant_cos_eur'] = (df_agg[COL_NB_COS] * df_agg[COL_MONTANT_MOY]) / 100.0

NOMS_MOIS = {1:'Jan',2:'Fev',3:'Mar',4:'Avr',5:'Mai',6:'Jun',
             7:'Jul',8:'Aou',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
df_agg['mois_nom'] = df_agg['mois'].map(NOMS_MOIS)

agg_mois = df_agg.groupby(['mois','mois_nom']).agg(
    nb_cos   = (COL_NB_COS,       'sum'),
    montant  = ('montant_cos_eur', 'sum'),
    captures = (COL_CAP_COS,      'sum'),
    taux_cap = (COL_TAUX_COS,     'mean'),
    nb_gab   = (COL_AUTOMATE_A,   'nunique'),
).reset_index().sort_values('mois')

agg_gab = df_agg.groupby(COL_AUTOMATE_A).agg(
    nb_cos      = (COL_NB_COS,       'sum'),
    montant_tot = ('montant_cos_eur', 'sum'),
    montant_moy = ('montant_cos_eur', 'mean'),
    captures    = (COL_CAP_COS,      'sum'),
    taux_cap    = (COL_TAUX_COS,     'mean'),
    nb_mois     = ('mois',            'nunique'),
).reset_index().sort_values('nb_cos', ascending=False)

agg_gab_mois = df_agg.groupby([COL_AUTOMATE_A,'mois','mois_nom']).agg(
    nb_cos   = (COL_NB_COS,       'sum'),
    montant  = ('montant_cos_eur', 'sum'),
    captures = (COL_CAP_COS,      'sum'),
    taux_cap = (COL_TAUX_COS,     'mean'),
).reset_index().sort_values([COL_AUTOMATE_A,'mois'])

print('Table agregee OK : {:,} GAB COS | {} mois'.format(
    df_agg[COL_AUTOMATE_A].nunique(), df_agg['mois'].nunique()))

## Comparaison brute vs agregee — validation croisee

In [ ]:
# ── 6. Comparaison brute vs agregee ─────────────────────────────────────────
# Les deux sources doivent donner des chiffres proches
# Un ecart important = signe que le filtre COS capture des choses differentes

comp = pd.DataFrame({
    'Indicateur': [
        'Nb operations COS',
        'Nb GAB concernes',
        'Montant total (EUR)',
        'Montant moyen (EUR)',
    ],
    'Table brute': [
        '{:,}'.format(int(agg_brut_gab['nb_cos'].sum())),
        '{:,}'.format(len(agg_brut_gab)),
        '{:,.0f}'.format(agg_brut_gab['montant_tot'].sum()),
        '{:.2f}'.format(agg_brut_gab['montant_moy'].mean()),
    ],
    'Table agregee': [
        '{:,}'.format(int(agg_gab['nb_cos'].sum())),
        '{:,}'.format(len(agg_gab)),
        '{:,.0f}'.format(agg_gab['montant_tot'].sum()),
        '{:.2f}'.format(agg_gab['montant_moy'].mean()),
    ],
})

display(comp.style
    .set_properties(**{'font-size':'12px','padding':'8px 14px'})
    .set_table_styles([{
        'selector':'th',
        'props':[('background-color','#003B5C'),('color','white'),
                 ('font-size','12px'),('padding','8px 14px')]
    }])
    .hide_index()
)
print('Si les chiffres sont proches -> les deux sources sont coherentes.')
print('Si ecart important -> verifier le filtre COS (colonne reseau vs privative).')

## KPIs globaux — source : table agregee

In [ ]:
# ── 7. KPIs globaux (source : table agregee) ────────────────────────────────
kpis = [
    ('Operations COS 2025',   '{:,}'.format(int(agg_gab['nb_cos'].sum()))),
    ('GAB avec activite COS', '{:,}'.format(len(agg_gab))),
    ('Montant estime (EUR)',   '{:,.0f}'.format(agg_gab['montant_tot'].sum())),
    ('Captures COS totales',  '{:,}'.format(int(agg_gab['captures'].sum()))),
    ('Taux capture moy.',     '{:.4f}%'.format(agg_gab['taux_cap'].mean())),
]
couleurs = ['#003B5C','#00B5AD','#E84E0F','#C0392B','#6A1B9A']
html = '<div style="display:flex;gap:14px;flex-wrap:wrap;margin:16px 0">'
for i,(label,val) in enumerate(kpis):
    html += (
        '<div style="background:{c};color:#fff;border-radius:10px;'
        'padding:18px 24px;min-width:160px;box-shadow:0 3px 10px rgba(0,0,0,.14)">'
        '<div style="font-size:28px;font-weight:bold;letter-spacing:-1px">{v}</div>'
        '<div style="font-size:11px;margin-top:6px;opacity:.88;'
        'text-transform:uppercase;letter-spacing:.5px">{l}</div></div>'
    ).format(c=couleurs[i],v=val,l=label)
html += '</div>'
display(HTML(html))

## Evolution mensuelle COS — table agregee

In [ ]:
# ── 8. Evolution mensuelle ──────────────────────────────────────────────────
fig_mois = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Nb retraits COS',
        'Montant total (EUR)',
        'Nb captures COS',
        'Taux de capture COS (%)',
    ],
    vertical_spacing=0.22, horizontal_spacing=0.12,
)

x = agg_mois['mois_nom'].tolist()

fig_mois.add_trace(go.Bar(
    x=x, y=agg_mois['nb_cos'].tolist(), marker_color='#003B5C',
    text=['{:,}'.format(int(v)) for v in agg_mois['nb_cos']],
    textposition='outside', textfont=dict(size=9), name='Nb COS',
), row=1, col=1)

fig_mois.add_trace(go.Bar(
    x=x, y=agg_mois['montant'].tolist(), marker_color='#00B5AD',
    text=['{:,.0f}'.format(v) for v in agg_mois['montant']],
    textposition='outside', textfont=dict(size=9), name='Montant',
), row=1, col=2)

fig_mois.add_trace(go.Bar(
    x=x, y=agg_mois['captures'].tolist(), marker_color='#C0392B',
    text=['{:,}'.format(int(v)) for v in agg_mois['captures']],
    textposition='outside', textfont=dict(size=9), name='Captures',
), row=2, col=1)

fig_mois.add_trace(go.Scatter(
    x=x, y=agg_mois['taux_cap'].tolist(),
    mode='lines+markers',
    line=dict(color='#E84E0F', width=2.5),
    marker=dict(size=9, color='#E84E0F', line=dict(color='white', width=1.5)),
    name='Taux capture',
), row=2, col=2)

fig_mois.update_layout(
    height=560, showlegend=False,
    title_text='Evolution mensuelle COS 2025',
    title_font=dict(size=14, color='#003B5C'),
    paper_bgcolor='#F7F9FC', plot_bgcolor='#FFFFFF',
)
fig_mois.update_xaxes(tickangle=-30, tickfont=dict(size=9))
fig_mois.show()

## Top 20 GAB — Volume COS (couleur = taux de capture)

In [ ]:
# ── 9. Top 20 GAB ───────────────────────────────────────────────────────────
top20 = agg_gab.head(20).copy()

fig_top = px.bar(
    top20,
    x='nb_cos', y=COL_AUTOMATE_A, orientation='h',
    color='taux_cap',
    color_continuous_scale=['#D5F5E3','#C0392B'],
    text='nb_cos',
    labels={
        'nb_cos':       'Nb retraits COS 2025',
        COL_AUTOMATE_A: 'Automate',
        'taux_cap':     'Taux capture COS (%)',
    },
    title='Top 20 GAB par volume COS 2025 — couleur = taux de capture',
)
fig_top.update_traces(
    texttemplate='%{text:,}', textposition='outside', textfont=dict(size=10),
)
fig_top.update_layout(
    height=560,
    yaxis=dict(autorange='reversed', tickfont=dict(size=9)),
    title_font=dict(size=14, color='#003B5C'),
    paper_bgcolor='#F7F9FC', plot_bgcolor='#FFFFFF',
    coloraxis_colorbar_title='Taux capture (%)',
)
fig_top.show()

## Volume COS vs Taux de capture — identifier les GAB prioritaires

In [ ]:
# ── 10. Scatter volume vs taux capture ──────────────────────────────────────
fig_sc = px.scatter(
    agg_gab,
    x='nb_cos', y='taux_cap',
    size='montant_tot', color='captures',
    color_continuous_scale=['#D6EAF8','#C0392B'],
    hover_name=COL_AUTOMATE_A,
    hover_data={
        'nb_cos':     ':,',
        'taux_cap':   ':.4f',
        'montant_tot':':.0f',
        'captures':   ':,',
        'nb_mois':    True,
    },
    labels={
        'nb_cos':     'Nb retraits COS 2025',
        'taux_cap':   'Taux de capture COS (%)',
        'montant_tot':'Montant total (EUR)',
        'captures':   'Nb captures',
        'nb_mois':    'Nb mois actifs',
    },
    title='Volume COS vs Taux de capture — haut-droite = prioritaire',
)
fig_sc.add_hline(y=agg_gab['taux_cap'].mean(), line_dash='dash',
    line_color='grey', annotation_text='Moy. taux capture',
    annotation_position='top right')
fig_sc.add_vline(x=agg_gab['nb_cos'].mean(), line_dash='dash',
    line_color='grey', annotation_text='Moy. volume COS',
    annotation_position='top right')
fig_sc.update_layout(
    height=480, title_font=dict(size=14, color='#003B5C'),
    paper_bgcolor='#F7F9FC', plot_bgcolor='#FFFFFF',
    coloraxis_colorbar_title='Nb captures',
)
fig_sc.show()

## Zoom interactif — suivre un GAB mois par mois

In [ ]:
# ── 11. Zoom interactif ─────────────────────────────────────────────────────
liste_gab = sorted(agg_gab[COL_AUTOMATE_A].tolist())

select_gab = widgets.Dropdown(
    options=liste_gab, value=liste_gab[0],
    description='Automate :',
    layout=widgets.Layout(width='320px'),
    style={'description_width': '80px'},
)
out_zoom = widgets.Output()

def afficher_gab(change):
    gab_id = select_gab.value
    df_g = agg_gab_mois[
        agg_gab_mois[COL_AUTOMATE_A] == gab_id
    ].sort_values('mois')

    fig_z = make_subplots(
        rows=1, cols=4,
        subplot_titles=['Nb COS','Montant (EUR)','Captures','Taux capture (%)']
    )
    mx = df_g['mois_nom'].tolist()
    for col,color,ci in [
        ('nb_cos',  '#003B5C',1),('montant','#00B5AD',2),
        ('captures','#C0392B',3),('taux_cap','#E84E0F',4),
    ]:
        fig_z.add_trace(go.Scatter(
            x=mx, y=df_g[col].tolist(), mode='lines+markers',
            line=dict(color=color, width=2.5),
            marker=dict(size=9, color=color, line=dict(color='white',width=1.5)),
            showlegend=False,
        ), row=1, col=ci)

    fig_z.update_layout(
        height=300, showlegend=False,
        title_text='GAB {} — detail mensuel COS 2025'.format(gab_id),
        title_font=dict(size=13, color='#003B5C'),
        paper_bgcolor='#F7F9FC', plot_bgcolor='#FFFFFF',
    )
    fig_z.update_xaxes(tickangle=-30, tickfont=dict(size=8))

    row_g = agg_gab[agg_gab[COL_AUTOMATE_A] == gab_id]
    fiche = ''
    if len(row_g) > 0:
        r = row_g.iloc[0]
        fiche = (
            '<div style="background:#003B5C;color:#fff;border-radius:8px;'
            'padding:10px 16px;margin-top:8px;font-size:12px">'
            '<b>{}</b> &nbsp;|&nbsp; {:,} retraits COS'
            ' &nbsp;|&nbsp; {:.0f} EUR total'
            ' &nbsp;|&nbsp; {:,} captures'
            ' &nbsp;|&nbsp; taux : {:.4f}%'
            ' &nbsp;|&nbsp; {} mois/12'
            '</div>'
        ).format(
            gab_id, int(r['nb_cos']), r['montant_tot'],
            int(r['captures']), r['taux_cap'], int(r['nb_mois']),
        )

    with out_zoom:
        out_zoom.clear_output(wait=True)
        fig_z.show()
        if fiche:
            display(HTML(fiche))

select_gab.observe(afficher_gab, names='value')
display(widgets.VBox([
    widgets.HTML(
        '<div style="margin:8px 0;font-weight:bold;color:#003B5C">'
        'Selectionnez un automate :</div>'
    ),
    select_gab, out_zoom,
]))
afficher_gab(None)

## Tableau recapitulatif complet

In [ ]:
# ── 12. Tableau recapitulatif ───────────────────────────────────────────────
recap = agg_gab.rename(columns={
    COL_AUTOMATE_A: 'Automate',
    'nb_cos':       'Nb retraits COS',
    'montant_tot':  'Montant total (EUR)',
    'montant_moy':  'Montant moy/mois (EUR)',
    'captures':     'Nb captures COS',
    'taux_cap':     'Taux capture (%)',
    'nb_mois':      'Nb mois actifs',
}).copy()

recap['Montant total (EUR)']    = recap['Montant total (EUR)'].round(0).astype(int)
recap['Montant moy/mois (EUR)'] = recap['Montant moy/mois (EUR)'].round(2)
recap['Taux capture (%)']       = recap['Taux capture (%)'].round(4)

seuil = recap['Nb retraits COS'].quantile(0.90)

def style_row(row):
    if row['Nb retraits COS'] >= seuil:
        return ['background-color:#FDEBD0'] * len(row)
    return [''] * len(row)

display(
    recap.style
    .apply(style_row, axis=1)
    .format({
        'Nb retraits COS':       '{:,}',
        'Montant total (EUR)':   '{:,}',
        'Montant moy/mois (EUR)':'{:.2f}',
        'Nb captures COS':       '{:,}',
        'Taux capture (%)':      '{:.4f}',
    })
    .set_properties(**{'font-size':'11px'})
    .set_table_styles([{
        'selector':'th',
        'props':[('background-color','#003B5C'),('color','white'),
                 ('font-size','11px'),('padding','6px 10px')]
    }])
    .hide_index()
)
print('Lignes orange = Top 10% volume COS (seuil : {:,} retraits)'.format(int(seuil)))